# Industrial Surface Defect Detection

Image classification pipeline that detects **5 classes** of industrial surface
condition: `crack`, `hole`, `normal`, `rust`, `scratch`.

**Approach:** Transfer learning (ImageNet-pretrained CNN backbone) with two-phase
fine-tuning, augmentation, native-resolution input, mixed precision, test-time
augmentation at evaluation, and network-failure-resilient weight loading.

## Before you run — two things that will stop this cold if missed

Both are one-click fixes in **Settings** (right sidebar):

1. **GPU**: Accelerator → **T4 x2** or **P100**. Without this, training runs on CPU
   and takes hours instead of minutes.
2. **Internet**: toggle **ON**. Needed once, to download the ImageNet-pretrained
   backbone (~20MB). Cell 1 below checks this automatically and warns you clearly if
   it's off. If you toggle it on mid-session, **restart the session** afterward —
   the change doesn't apply retroactively to an already-running kernel.

If instead the very first `import torch` cell fails with an `AttributeError` mentioning
`torch.fx` and "circular import" — that's a broken PyTorch build in your current Kaggle
kernel, not this notebook's code (it fails inside PyTorch's own package init, before any
of this notebook's code runs). Restart the session (or Factory Reset); if it persists,
run `!pip install -U torch --quiet` in a new cell, restart again, and re-run.

---


In [ ]:
# 1. Environment check (GPU + internet, since both commonly trip up a fresh Kaggle session)
import os, sys, json, random, time, copy, socket
import numpy as np

try:
    import torch
except AttributeError as e:
    if "partially initialized module" in str(e) or "circular import" in str(e):
        print("=" * 72)
        print("PyTorch failed to import due to a broken build in this Kaggle session")
        print("(a circular import inside torch's own package init) — this is NOT a bug")
        print("in this notebook's code; nothing here has run yet.")
        print("Fix: Kernel menu -> Restart Session (or Factory Reset), then Run All again.")
        print("If it recurs, run '!pip install -U torch --quiet' in a new cell, restart")
        print("the session once more, and re-run.")
        print("=" * 72)
    raise

import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms, models
from torchvision.datasets import ImageFolder

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"Torchvision  : {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Settings > Accelerator and enable a "
          "GPU (T4 x2 or P100) — training on CPU will be very slow.")

def check_internet(host="download.pytorch.org", port=443, timeout=3):
    try:
        socket.create_connection((host, port), timeout=timeout)
        return True
    except OSError:
        return False

internet_ok = check_internet()
print(f"Internet access: {'OK' if internet_ok else 'NOT AVAILABLE'}")
if not internet_ok:
    print("WARNING: No internet access detected. Downloading ImageNet-pretrained weights")
    print("will fail. Go to Settings > Internet > toggle ON, then RESTART the session.")
    print("Training will still run without it (falls back to random init further down),")
    print("but accuracy will be substantially lower without pretrained features.")


## 2. Configuration

All tunable settings live here. Defaults are tuned for a balanced, ~12k-image, 5-class, 256x256 grayscale dataset on a Kaggle GPU session, with a longer fine-tuning schedule aimed at squeezing out the last percentage points of accuracy.

In [ ]:
# 2. Config
SEED = 42

# Set this manually ONLY if auto-detection below fails, e.g.:
# MANUAL_DATA_DIR = "/kaggle/input/<your-dataset-slug>/industrial_defect_dataset"
MANUAL_DATA_DIR = None

IMG_SIZE      = 256          # native resolution of this dataset — no resize needed,
                              # which preserves fine detail in thin cracks/scratches
BATCH_SIZE    = 32
NUM_WORKERS   = 2

ARCH          = "efficientnet_b0"   # one of: efficientnet_b0, resnet50, resnet18
PHASE1_EPOCHS = 5             # frozen backbone, train classifier head only
PHASE2_EPOCHS = 30            # unfrozen, full fine-tune (early stopping usually cuts this shorter)
PHASE1_LR     = 1e-3
PHASE2_LR     = 1e-4
WEIGHT_DECAY  = 1e-4
LABEL_SMOOTHING = 0.1
EARLY_STOP_PATIENCE = 8

USE_TTA = True   # test-time augmentation (avg over original/h-flip/v-flip) for final eval

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_device_type = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = (device.type == "cuda")

OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_model.pth")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
torch.backends.cudnn.benchmark = True
print(f"Device: {device} | Mixed precision: {use_amp} | Output dir: {OUTPUT_DIR}")


## 3. Locate the dataset

Searches `/kaggle/input` for a folder that directly contains `train` and `val` subfolders, so this cell works regardless of the exact dataset slug Kaggle assigns on upload, and de-duplicates in case search roots overlap.

In [ ]:
# 3. Locate dataset (robust to whatever slug Kaggle gives your uploaded dataset)
def find_dataset_roots():
    found, seen = [], set()
    for base in ["/kaggle/input", "/kaggle/working", "."]:
        if not os.path.isdir(base):
            continue
        for root, dirs, _files in os.walk(base):
            if "train" in dirs and "val" in dirs:
                abs_root = os.path.abspath(root)
                if abs_root not in seen:
                    seen.add(abs_root)
                    found.append(root)
    return found

if MANUAL_DATA_DIR:
    DATA_DIR = MANUAL_DATA_DIR
else:
    roots = find_dataset_roots()
    DATA_DIR = roots[0] if roots else None

if not DATA_DIR or not os.path.isdir(DATA_DIR):
    print("Could not automatically locate the dataset.")
    if os.path.isdir("/kaggle/input"):
        print("Contents of /kaggle/input:")
        for item in os.listdir("/kaggle/input"):
            print(" -", item)
    print('\nSet MANUAL_DATA_DIR above to the correct path, e.g.:')
    print('MANUAL_DATA_DIR = "/kaggle/input/<your-dataset-slug>/industrial_defect_dataset"')
    raise FileNotFoundError("No directory containing both 'train' and 'val' subfolders was found.")

train_dir = os.path.join(DATA_DIR, "train")
val_dir   = os.path.join(DATA_DIR, "val")
print(f"Dataset root : {DATA_DIR}")
print(f"Train dir    : {train_dir}")
print(f"Val dir      : {val_dir}")


## 4. Dataset exploration

Sanity-check class balance and image counts before training — catches accidental empty folders or lopsided splits early.

In [ ]:
# 4. Explore class balance
def count_images(split_dir):
    counts = {}
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if os.path.isdir(cls_dir):
            counts[cls] = len([f for f in os.listdir(cls_dir)
                                if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))])
    return counts

train_counts = count_images(train_dir)
val_counts   = count_images(val_dir)

print(f"{'Class':<12}{'Train':>10}{'Val':>10}")
for cls in train_counts:
    print(f"{cls:<12}{train_counts.get(cls,0):>10}{val_counts.get(cls,0):>10}")
print(f"{'TOTAL':<12}{sum(train_counts.values()):>10}{sum(val_counts.values()):>10}")

fig, ax = plt.subplots(figsize=(8, 5))
classes_sorted = list(train_counts.keys())
x = np.arange(len(classes_sorted))
width = 0.35
ax.bar(x - width/2, [train_counts[c] for c in classes_sorted], width, label="Train")
ax.bar(x + width/2, [val_counts[c] for c in classes_sorted], width, label="Val")
ax.set_xticks(x); ax.set_xticklabels(classes_sorted)
ax.set_ylabel("Number of images")
ax.set_title("Class distribution")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "class_distribution.png"), dpi=130)
plt.show()


In [ ]:
# 4b. Preview sample images from each class
fig, axes = plt.subplots(1, len(classes_sorted), figsize=(3 * len(classes_sorted), 3.2))
for ax, cls in zip(axes, classes_sorted):
    cls_dir = os.path.join(train_dir, cls)
    fname = sorted(os.listdir(cls_dir))[0]
    img = plt.imread(os.path.join(cls_dir, fname))
    ax.imshow(img, cmap="gray")
    ax.set_title(cls)
    ax.axis("off")
plt.suptitle("Sample image per class")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sample_preview.png"), dpi=130)
plt.show()


## 5. Data loading & augmentation

Images are used at their native 256x256 resolution (no resize) to preserve fine detail in thin cracks/scratches. `ImageFolder`'s default loader converts grayscale to 3-channel RGB automatically (required for ImageNet-pretrained backbones). Train-time augmentation includes flips and rotation — reasonable here because a surface defect has no canonical orientation. Normalization uses standard ImageNet statistics to match backbone pretraining.

In [ ]:
# 5. Transforms & DataLoaders
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),   # no-op if images are already IMG_SIZE, but
    transforms.RandomHorizontalFlip(p=0.5),    # safe if a stray image differs
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = ImageFolder(train_dir, transform=train_transforms)
val_dataset   = ImageFolder(val_dir, transform=val_transforms)

assert train_dataset.classes == val_dataset.classes, "Train/val class folders don't match!"
class_names = train_dataset.classes
num_classes = len(class_names)
print(f"Classes ({num_classes}): {class_names}")
print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
                           drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))

# Sanity check a single batch
imgs, labels = next(iter(train_loader))
print(f"Batch shape: {imgs.shape} | Label range: {labels.min().item()}-{labels.max().item()}")


## 6. Model — transfer learning backbone

Loads an ImageNet-pretrained backbone and replaces its classifier head for our 5 classes. Handles both the modern `weights=` torchvision API and the older `pretrained=` boolean. **If the weight download fails for any reason (e.g. Internet is off), this falls back to a randomly-initialized backbone with a clear warning instead of crashing** — training still proceeds, just from a weaker starting point.

In [ ]:
# 6. Model factory (transfer learning, network-failure-resilient)
def _instantiate(builder, weights):
    """Call builder with the modern weights= kwarg; fall back to the legacy
    pretrained= boolean for older torchvision installs."""
    try:
        return builder(weights=weights)
    except TypeError:
        return builder(pretrained=(weights is not None))


def build_model(arch, num_classes, pretrained=True):
    arch = arch.lower()
    registry = {
        "efficientnet_b0": (models.efficientnet_b0, "EfficientNet_B0_Weights", "IMAGENET1K_V1"),
        "resnet50":        (models.resnet50,        "ResNet50_Weights",        "IMAGENET1K_V2"),
        "resnet18":        (models.resnet18,        "ResNet18_Weights",        "IMAGENET1K_V1"),
    }
    if arch not in registry:
        raise ValueError(f"Unknown ARCH '{arch}'. Choose from {list(registry)}.")
    builder, weights_cls_name, weights_member = registry[arch]

    model, used_pretrained = None, False

    if pretrained:
        try:
            weights_enum = getattr(getattr(models, weights_cls_name), weights_member)
        except AttributeError:
            weights_enum = None   # very old torchvision without the Weights-enum API
        try:
            model = _instantiate(builder, weights_enum)
            used_pretrained = True
        except Exception as e:
            print(f"  NOTE: could not download pretrained weights for '{arch}' "
                  f"({type(e).__name__}: {str(e)[:150]}).")
            print("  Continuing with random initialization instead. For much better")
            print("  accuracy, enable Internet (Settings > Internet), restart the")
            print("  session, and re-run this notebook.")

    if model is None:
        model = _instantiate(builder, None)
        used_pretrained = False

    if arch == "efficientnet_b0":
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model, used_pretrained


def get_head_params(model, arch):
    return model.classifier.parameters() if arch.lower() == "efficientnet_b0" else model.fc.parameters()


def set_backbone_frozen(model, arch, freeze):
    for p in model.parameters():
        p.requires_grad = not freeze
    for p in get_head_params(model, arch):   # head always trainable
        p.requires_grad = True


model, used_pretrained = build_model(ARCH, num_classes, pretrained=True)
model = model.to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Architecture: {ARCH}")
print(f"Pretrained weights loaded: {used_pretrained}")
print(f"Total parameters: {total_params:,}")
if not used_pretrained:
    print("Training from random initialization — expect noticeably lower accuracy than")
    print("with pretrained weights. Fix Internet access and re-run for the intended result.")


## 7. Training utilities

Mixed precision is enabled only when a CUDA GPU is present. A CUDA out-of-memory error, if it happens, is caught once to print an actionable message (lower `BATCH_SIZE`) instead of a bare stack trace.

In [ ]:
# 7. Train / eval loops + early stopping
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

# Modern torch.amp API (torch.cuda.amp.* is deprecated as of PyTorch 2.4+), with a
# fallback to the legacy API for older PyTorch installs.
try:
    scaler = torch.amp.GradScaler(amp_device_type, enabled=use_amp)
    def autocast_ctx():
        return torch.amp.autocast(amp_device_type, enabled=use_amp)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    def autocast_ctx():
        return torch.cuda.amp.autocast(enabled=use_amp)


def _is_oom_error(e):
    return isinstance(e, RuntimeError) and "out of memory" in str(e).lower()


def train_one_epoch(model, loader, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    try:
        for images, labels in loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast_ctx():
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * images.size(0)
            correct += outputs.argmax(1).eq(labels).sum().item()
            total += labels.size(0)
    except RuntimeError as e:
        if _is_oom_error(e):
            if device.type == "cuda":
                torch.cuda.empty_cache()
            print(f"\nCUDA out of memory at batch size {BATCH_SIZE}. Lower BATCH_SIZE in "
                  f"the config cell (try 16 or 8), then restart the session and re-run.")
        raise
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with autocast_ctx():
            outputs = model(images)
            loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())
    return running_loss / total, correct / total, all_preds, all_labels


@torch.no_grad()
def evaluate_tta(model, loader, tta=True):
    """Final-evaluation metric: averages softmax probabilities over the original image
    plus horizontal- and vertical-flipped views. Not used during per-epoch training
    validation (kept fast there) — only for the reported final number and inference."""
    model.eval()
    flip_dims = [None, 3, 2] if tta else [None]   # None=original, 3=h-flip, 2=v-flip (NCHW)
    all_probs, all_labels = [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        probs_sum = 0.0
        for dim in flip_dims:
            aug = images if dim is None else torch.flip(images, dims=[dim])
            with autocast_ctx():
                outputs = model(aug)
            probs_sum = probs_sum + torch.softmax(outputs.float(), dim=1)
        all_probs.append((probs_sum / len(flip_dims)).cpu())
        all_labels.extend(labels.numpy().tolist())
    all_probs_t = torch.cat(all_probs, dim=0)
    preds = all_probs_t.argmax(1).numpy().tolist()
    acc = accuracy_score(all_labels, preds)
    return acc, preds, all_labels, all_probs_t.numpy()


class EarlyStopper:
    def __init__(self, patience=8, mode="max"):
        self.patience, self.mode = patience, mode
        self.best, self.counter, self.should_stop = None, 0, False

    def step(self, value):
        improved = self.best is None or (value > self.best if self.mode == "max" else value < self.best)
        if improved:
            self.best, self.counter = value, 0
        else:
            self.counter += 1
            self.should_stop = self.counter >= self.patience
        return improved


history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": [], "phase": []}
early_stopper = EarlyStopper(patience=EARLY_STOP_PATIENCE, mode="max")

def run_phase(optimizer, scheduler, num_epochs, phase_name):
    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
        val_loss, val_acc, _, _ = evaluate(model, val_loader)

        if scheduler is not None:
            scheduler.step()

        current_lr = optimizer.param_groups[0]["lr"]
        history["train_loss"].append(train_loss); history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss); history["val_acc"].append(val_acc)
        history["lr"].append(current_lr); history["phase"].append(phase_name)

        improved = early_stopper.step(val_acc)
        marker = ""
        if improved:
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            marker = "  <-- best so far, checkpoint saved"

        print(f"[{phase_name}] Epoch {epoch:>2}/{num_epochs} | "
              f"Train loss {train_loss:.4f} acc {train_acc:.4f} | "
              f"Val loss {val_loss:.4f} acc {val_acc:.4f} | "
              f"LR {current_lr:.2e} | {time.time()-t0:.1f}s{marker}")

        if early_stopper.should_stop:
            print(f"Early stopping: no val-accuracy improvement for {EARLY_STOP_PATIENCE} epochs.")
            return True
    return False


## 8. Phase 1 — train the classifier head (frozen backbone)

Warms up the new classification head with the pretrained backbone frozen, so early large gradients from a randomly-initialized head don't destroy pretrained features. (If pretrained weights failed to download, this still runs — it's just warming up a head on top of a random backbone.)

In [ ]:
# 8. Phase 1: frozen backbone, head warmup
set_backbone_frozen(model, ARCH, freeze=True)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters (head only): {trainable:,} / {total_params:,}")

optimizer1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                lr=PHASE1_LR, weight_decay=WEIGHT_DECAY)
scheduler1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer1, T_max=PHASE1_EPOCHS)

stopped = run_phase(optimizer1, scheduler1, PHASE1_EPOCHS, "phase1-head")


## 9. Phase 2 — fine-tune the entire network

Unfreezes all layers and continues training at a lower learning rate so the backbone can adapt its features to this specific texture domain without catastrophic forgetting. Given 30 epochs of budget here with early stopping, this phase is where most of the accuracy gains toward a high final number happen.

In [ ]:
# 9. Phase 2: full fine-tune
if not early_stopper.should_stop:
    set_backbone_frozen(model, ARCH, freeze=False)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters (full network): {trainable:,} / {total_params:,}")

    optimizer2 = torch.optim.AdamW(model.parameters(), lr=PHASE2_LR, weight_decay=WEIGHT_DECAY)
    scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=PHASE2_EPOCHS)

    run_phase(optimizer2, scheduler2, PHASE2_EPOCHS, "phase2-finetune")
else:
    print("Skipping phase 2 — early stopping already triggered in phase 1.")

print(f"\nBest single-pass validation accuracy during training: {early_stopper.best*100:.2f}%")
print("(Final reported accuracy below uses test-time augmentation and may be higher.)")


## 10. Training curves

In [ ]:
# 10. Plot loss & accuracy curves
epochs_range = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history["train_loss"], marker="o", markersize=3, label="Train")
axes[0].plot(epochs_range, history["val_loss"], marker="o", markersize=3, label="Val")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history["train_acc"], marker="o", markersize=3, label="Train")
axes[1].plot(epochs_range, history["val_acc"], marker="o", markersize=3, label="Val")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].set_title("Accuracy")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=130)
plt.show()


## 11. Final evaluation (best checkpoint, with test-time augmentation)

Loads the best checkpoint from training and evaluates with TTA (average of original, horizontal-flip, and vertical-flip predictions) — typically a small but reliable accuracy boost over a single forward pass, for free at inference time.

In [ ]:
# 11. Load best checkpoint and evaluate fully (with TTA)
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))

val_acc_tta, all_preds, all_labels, all_probs = evaluate_tta(model, val_loader, tta=USE_TTA)
print(f"Final validation accuracy {'(with TTA)' if USE_TTA else '(single-pass)'}: {val_acc_tta*100:.2f}%\n")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))


In [ ]:
# 11b. Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax,
            cbar_kws={"label": "Count"})
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix — Val Accuracy {val_acc_tta*100:.2f}%")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=130)
plt.show()


## 12. Sample predictions

In [ ]:
# 12. Visualize a batch of predictions (green=correct, red=incorrect)
mean_t = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std_t  = torch.tensor(IMAGENET_STD).view(3, 1, 1)

sample_idx = random.sample(range(len(val_dataset)), 10)
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
model.eval()
with torch.no_grad():
    for ax, idx in zip(axes.flat, sample_idx):
        img_tensor, label = val_dataset[idx]
        output = model(img_tensor.unsqueeze(0).to(device))
        pred = output.argmax(1).item()

        img_show = (img_tensor * std_t + mean_t).permute(1, 2, 0).clamp(0, 1).numpy()
        ax.imshow(img_show)
        ax.axis("off")
        color = "green" if pred == label else "red"
        ax.set_title(f"True: {class_names[label]}\nPred: {class_names[pred]}", color=color, fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sample_predictions.png"), dpi=130)
plt.show()


## 13. Inference on a new image

Reusable function for predicting on any single image outside the dataset (e.g. a new factory photo), using the same TTA strategy as the final evaluation above.

In [ ]:
# 13. Single-image inference helper (TTA-aware)
from PIL import Image

def predict_image(image_path, model=model, class_names=class_names, transform=val_transforms, tta=USE_TTA):
    model.eval()
    img = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)
    flip_dims = [None, 3, 2] if tta else [None]
    with torch.no_grad():
        probs_sum = 0.0
        for dim in flip_dims:
            aug = tensor if dim is None else torch.flip(tensor, dims=[dim])
            probs_sum = probs_sum + torch.softmax(model(aug).float(), dim=1)
        probs = (probs_sum / len(flip_dims))[0]
    pred_idx = int(probs.argmax().item())
    return {
        "predicted_class": class_names[pred_idx],
        "confidence": float(probs[pred_idx]),
        "all_probabilities": {c: float(probs[i]) for i, c in enumerate(class_names)},
    }

# Example (uncomment and point at any image path to try it):
# result = predict_image(os.path.join(val_dir, class_names[0], os.listdir(os.path.join(val_dir, class_names[0]))[0]))
# print(result)


## 14. Save artifacts

Saves the model weights, class mapping, and training history to `/kaggle/working` so they appear in the notebook's Output tab and can be downloaded or used in another notebook. Also records whether pretrained weights and TTA were actually used, since both materially affect the accuracy achieved.

In [ ]:
# 14. Persist everything
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "final_model.pth"))
with open(os.path.join(OUTPUT_DIR, "class_names.json"), "w") as f:
    json.dump(class_names, f, indent=2)
with open(os.path.join(OUTPUT_DIR, "training_history.json"), "w") as f:
    json.dump(history, f, indent=2)
with open(os.path.join(OUTPUT_DIR, "run_summary.json"), "w") as f:
    json.dump({
        "architecture": ARCH,
        "pretrained_weights_used": used_pretrained,
        "img_size": IMG_SIZE,
        "tta_used_for_final_eval": USE_TTA,
        "final_val_accuracy": val_acc_tta,
        "best_single_pass_val_accuracy_during_training": early_stopper.best,
        "epochs_run": len(history["train_loss"]),
    }, f, indent=2)

print("Saved to", OUTPUT_DIR)
print(" -", "best_model.pth, final_model.pth")
print(" -", "class_names.json, training_history.json, run_summary.json")
print(" -", "class_distribution.png, sample_preview.png, training_curves.png,")
print(" -", "confusion_matrix.png, sample_predictions.png")
print(f"\nFinal validation accuracy: {val_acc_tta*100:.2f}%  (pretrained weights used: {used_pretrained})")


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import torch
import os

# Change this path to your uploaded test image
image_path = "/kaggle/input/datasets/rahulamgoth/dataset_name/dataset_image.png"

img = Image.open(image_path).convert("RGB")

plt.figure(figsize=(6,6))
plt.imshow(img)
plt.axis("off")
plt.show()

# Apply validation transforms
img_tensor = val_transforms(img).unsqueeze(0).to(device)

model.eval()

with torch.no_grad():
    output = model(img_tensor)
    probs = torch.softmax(output, dim=1)

confidence, pred = torch.max(probs,1)

print("="*50)
print("Prediction :", class_names[pred.item()])
print("Confidence : {:.2f}%".format(confidence.item()*100))
print("="*50)

print("\nClass Probabilities\n")

for i, cls in enumerate(class_names):
    print(f"{cls:10s} : {probs[0][i].item()*100:.2f}%")
